<a href="https://colab.research.google.com/github/korkutanapa/OEE_ARTICLE_STUDIES/blob/main/TS_TDA_FEATURE_SELECTION_STEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
# Load the file for time series
data_path = '/content/tda_features_from_residuals_selected_scaled.xlsx'
df = pd.read_excel(data_path)

RFE

In [2]:
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.utils import resample

import warnings
import multiprocessing
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")



# Load dataset

target_column = "Target"
exog_columns = [col for col in df.columns if col != target_column]

# Split into training
train_df = df.iloc[:-4]
X_train = train_df[exog_columns]
y_train = train_df[target_column]

# SARIMAX p-value extractor
def get_p_values(X, y):
    try:
        model = SARIMAX(y,
                        order=(1, 0, 0),
                        seasonal_order=(1, 0, 1, 8),
                        exog=X)
        result = model.fit(disp=False)
        return result.pvalues.loc[X.columns]
    except Exception as e:
        print("SARIMAX fit error:", e)
        return pd.Series([1.0] * X.shape[1], index=X.columns)

# Model AIC calculator
def compute_aic(feature_set):
    try:
        model = SARIMAX(y_train,
                        order=(4, 0, 0),
                        seasonal_order=(1, 0, 1, 8),
                        exog=X_train[list(feature_set)])
        result = model.fit(disp=False)
        return (feature_set, result.aic)
    except:
        return (feature_set, np.inf)

# RFE parameters
selected_features = exog_columns.copy()
step2 = 5

# RFE Loop
while True:
    p_values = get_p_values(X_train[selected_features], y_train)

    # Features with p > 0.05
    worst_candidates = p_values[p_values > 0.05]

    # Stopping criterion
    if len(worst_candidates) < step2:
        print(f"Stopping early: Only {len(worst_candidates)} features have p > 0.05.")
        break

    # Dynamic step size
    step = min(20, len(worst_candidates), len(selected_features) // 2)

    # Worst features sorted by p-value
    worst_features = worst_candidates.sort_values(ascending=False).head(step).index

    for feature in worst_features:
        if feature in selected_features:
            selected_features.remove(feature)
            print(f"Removed Feature: {feature}, p-value: {p_values[feature]}")

print("\nInitial RFE Completed.")
print(f"Remaining Features: {len(selected_features)}")

# Optional: Stability Check via Bootstrapping
print("\nRunning stability check (bootstrap p-values)...")
bootstrap_rounds = 10
stable_features = []

boot_results = []
for i in range(bootstrap_rounds):
    X_res, y_res = resample(X_train[selected_features], y_train)
    p_vals = get_p_values(X_res, y_res)
    significant = set(p_vals[p_vals < 0.05].index)
    boot_results.append(significant)

stable_features = list(set.intersection(*boot_results))
print(f"\nStable Features Across {bootstrap_rounds} Bootstraps: {len(stable_features)}")

# Optional: AIC comparison for final selection
print("\nEvaluating AIC on stable subsets...")
feature_sets = [stable_features, selected_features]  # Compare both
num_cores = multiprocessing.cpu_count()

aic_values = Parallel(n_jobs=num_cores)(
    delayed(compute_aic)(f_set) for f_set in feature_sets
)

best_features, best_aic = sorted(aic_values, key=lambda x: x[1])[0]
print(f"\nBest AIC: {best_aic}")
print("Selected Final Features:", best_features)

Removed Feature: betti_var_H0, p-value: 0.9940962543292472
Removed Feature: dim_0_std_lifetime, p-value: 0.984340944320757
Removed Feature: HK_symmetry_F_H1, p-value: 0.9625011650951258
Removed Feature: dim_0_top3_lifetime_sum, p-value: 0.8499381142182753
Removed Feature: silhouette_skew_H1, p-value: 0.8380104884562659
Removed Feature: silhouette_kurt_H1, p-value: 0.8278288975119391
Removed Feature: amplitude_wasserstein_0, p-value: 0.7924162520589928
Removed Feature: dim_0_median_lifetime, p-value: 0.7847545781624754
Removed Feature: HK_entropy_H0, p-value: 0.7476648858073818
Removed Feature: dim_0_IQR_lifetime, p-value: 0.7065260080716537
Removed Feature: betti_skew_H0, p-value: 0.6897469879235418
Removed Feature: entropy_0, p-value: 0.6749881988505718
Removed Feature: HK_entropy_H1, p-value: 0.6637928794867032
Removed Feature: dim_0_carlsson_f2, p-value: 0.6499632007160037
Removed Feature: landscape_nonzero_count_H0, p-value: 0.6112820320238854
Removed Feature: dim_0_CV_lifetime, p-

In [3]:
columns_to_keep = selected_features.copy()  # Make a copy of your list
if 'Target' not in columns_to_keep:
    columns_to_keep.append('Target')

data_rfe = df[columns_to_keep]

In [4]:
data_rfe

,betti_L1_H1,betti_L2_H1,betti_max_H1,betti_skew_H1,betti_kurt_H1,betti_gini_H1,HK_col_L1_std_H0,HK_symmetry_F_H0,HK_Laplacian_trace_H0,HK_Laplacian_fro_H0,HK_Laplacian_eig_min_H1,HK_Laplacian_eig_max_H1,HK_Laplacian_fro_H1,Target
0,0.0,0.0,0.0,0.0,0.0,0.0,0.827921,0.792343,0.687299,0.787785,1.0,0.0,0.0,-36.630516
1,0.0,0.0,0.0,0.0,0.0,0.0,0.821216,0.670740,0.788977,0.785140,1.0,0.0,0.0,-32.755274
2,0.0,0.0,0.0,0.0,0.0,0.0,0.833436,0.702709,0.776120,0.795201,1.0,0.0,0.0,-16.242944
3,0.0,0.0,0.0,0.0,0.0,0.0,0.857574,0.659497,0.877793,0.827614,1.0,0.0,0.0,6.891582
4,0.0,0.0,0.0,0.0,0.0,0.0,0.781484,0.477937,0.896239,0.756582,1.0,0.0,0.0,-2.448256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
654,0.0,0.0,0.0,0.0,0.0,0.0,0.753624,0.514306,0.998906,0.774196,1.0,0.0,0.0,0.412770
655,0.0,0.0,0.0,0.0,0.0,0.0,0.790622,0.609572,1.000000,0.811783,1.0,0.0,0.0,0.543904
656,0.0,0.0,0.0,0.0,0.0,0.0,0.764561,0.526593,0.994675,0.781398,1.0,0.0,0.0,18.180364
657,0.0,0.0,0.0,0.0,0.0,0.0,0.721073,0.414802,0.994681,0.738514,1.0,0.0,0.0,13.045591


In [5]:
file_path = 'tda_features_from_residuals_selected_scaled_byrfe.xlsx'
data_rfe.to_excel(file_path, index=False)  # index=False to exclude the index column

PSO

In [ ]:
%pip install pyswarm


In [6]:
import numpy as np
import pandas as pd
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pyswarm import pso  # Particle Swarm Optimization
from functools import lru_cache
from collections import Counter
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Suppress warnings
warnings.filterwarnings("ignore")

# ============================
# Data Preparation
# ============================

target_column = 'Target'  # Replace with actual target column
train_data = data_rfe[:-4]  # Use all but the last 4 rows for training

selected_columns_train = train_data.drop(columns=[target_column])  # Features only
exog_train = selected_columns_train.values

num_features = selected_columns_train.shape[1]  # Number of features

# ============================
# Caching for Faster Execution
# ============================

solution_cache = {}

@lru_cache(maxsize=1000)
def evaluate_solution(solution_key):
    """
    Evaluates SARIMAX performance for a given feature subset using negative BIC.
    """
    selected_features = np.array(solution_key, dtype=bool)

    # Penalize solutions that select no features
    if not np.any(selected_features):
        return 1e6

    exog_selected = exog_train[:, selected_features]

    try:
        model = SARIMAX(train_data[target_column],
                        order=(1, 0, 0),
                        seasonal_order=(1, 0, 1, 8),
                        exog=exog_selected,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        results = model.fit(disp=0, maxiter=30)  # Lower maxiter for speed
        return results.bic  # Minimize BIC
    except:
        return 1e6  # Strong penalty for errors

# ============================
# Define PSO Objective Function
# ============================

def pso_objective_function(solution):
    """
    Evaluates the fitness of a feature subset for PSO.
    """
    key = tuple(np.round(solution).astype(int))  # Convert to binary feature selection

    if key in solution_cache:
        return solution_cache[key]

    # Evaluate solution
    fitness_val = evaluate_solution(key)
    solution_cache[key] = fitness_val  # Store in cache
    return fitness_val

# ============================
# PSO Execution
# ============================

lb = [0] * num_features  # Lower bounds (0 = feature not selected)
ub = [1] * num_features  # Upper bounds (1 = feature selected)

selected_features_runs = []
best_results = []
num_columns = data_rfe.shape[1]-1
for run in range(5):
    print(f"Starting PSO optimization, run {run+1}...")

    best_solution, best_bic = best_solution, best_bic = pso(
    pso_objective_function,
    lb,
    ub,
    swarmsize=40,
    maxiter=100,
    omega=0.7,  # Inertia weight
    phip=1.4,   # Cognitive parameter (similar to c1)
    phig=1.8,   # Social parameter (similar to c2)
    debug=False
)


    # Convert solution to selected feature names
    selected_features_mask = np.round(best_solution).astype(int).astype(bool)
    selected_features_pso = selected_columns_train.columns[selected_features_mask].tolist()
    selected_features_runs.append(selected_features_pso)

    # Fit final model with selected features
    exog_train_final = exog_train[:, selected_features_mask]


    final_model = SARIMAX(train_data[target_column],
                          order=(1, 0, 0),
                          seasonal_order=(1, 0, 1, 8),
                          exog=exog_train_final)
    final_results = final_model.fit(disp=False)



    # Save best results
    best_results.append({
        "Run": run + 1,
        "BIC": best_bic,
        "Selected Features": selected_features_pso
    })

# Convert to DataFrame
results_df = pd.DataFrame(best_results)

# Identify the best run (minimum BIC)
best_run = results_df.loc[results_df[['BIC']].sum(axis=1).idxmin()]
best_run_features = best_run["Selected Features"]

# Find the most frequently chosen features (chosen in at least 3 runs)
feature_counts = Counter(feature for run_features in selected_features_runs for feature in run_features)
final_feature_selection = list(feature_counts.keys())

# If final_feature_selection is empty, assign it to best_run_features
if not final_feature_selection:
    final_feature_selection = best_run_features

# Save best run features and final feature selection in separate lists
best_run_feature_list = list(best_run_features)
final_feature_selection_list = list(final_feature_selection)

# ============================
# Display Results
# ============================

print("\nPSO Runs Results:")
print(results_df)

print("\nBest Run Summary:")
print(best_run)

print("\nFinal Feature Selection (Chosen in at least 3 runs or from best run if empty):")
print(final_feature_selection_list)

print("\nBest Run Features List:")
print(best_run_feature_list)

# ============================
# Save Results
# ============================

# Convert to DataFrame for easy export if needed
best_run_summary_df = pd.DataFrame([best_run])
final_features_df = pd.DataFrame({"Most Chosen Features": final_feature_selection_list})


Starting PSO optimization, run 1...
Stopping search: maximum iterations reached --> 100
Starting PSO optimization, run 2...
Stopping search: maximum iterations reached --> 100
Starting PSO optimization, run 3...
Stopping search: maximum iterations reached --> 100
Starting PSO optimization, run 4...
Stopping search: maximum iterations reached --> 100
Starting PSO optimization, run 5...
Stopping search: maximum iterations reached --> 100

PSO Runs Results:
   Run          BIC   Selected Features
0    1  4561.044087     [betti_kurt_H1]
1    2  4561.044087     [betti_kurt_H1]
2    3  4562.141277     [betti_skew_H1]
3    4  4562.141277     [betti_skew_H1]
4    5  4563.565306  [HK_symmetry_F_H0]

Best Run Summary:
Run                                1
BIC                      4561.044087
Selected Features    [betti_kurt_H1]
Name: 0, dtype: object

Final Feature Selection (Chosen in at least 3 runs or from best run if empty):
['betti_kurt_H1', 'betti_skew_H1', 'HK_symmetry_F_H0']

Best Run Fea

In [7]:
# Save best run features and final feature selection in separate lists
best_run_feature_list = list(best_run_features)
final_feature_selection_list = list(final_feature_selection)

In [8]:
# Convert selected_features_pso to a standard Python list
selected_features_pso = best_run_feature_list

# Add 'Target' to the selected features list
selected_features_pso.append('Target')

In [9]:
data_pso=data_rfe[selected_features_pso]

In [10]:
# Convert selected_features_pso to a standard Python list
selected_features_pso = final_feature_selection_list

# Add 'Target' to the selected features list
selected_features_pso.append('Target')

In [11]:
data_pso_best=data_rfe[selected_features_pso]

In [12]:
file_path = 'tda_features_from_residuals_selected_scaled_bypso.xlsx'
data_pso_best.to_excel(file_path, index=False)  # index=False to exclude the index column

In [13]:
file_path = 'tda_features_from_residuals_selected_scaled_bypso_2.xlsx'
data_pso.to_excel(file_path, index=False)  # index=False to exclude the index column